# 📋 Step 3: 의회별 파싱 규칙 검증 및 통합 규칙 JSON 생성
2단계에서 도출된 개별 의회별 규칙 파일(`rule_*.json`)들을 취합하여,
선택자의 문법적 유효성과 한글 오염 여부를 최종 검증하고, 결측 시 안전한 표준 Fallback을 적용하여
단일 통합 파싱 규칙 파일(`consolidated_parsing_rules_2024.json`)로 병합합니다.

### 📌 주요 처리 과정
1. **규칙 파일 수집 (`rule_*.json`)**: 생성된 모든 개별 의회 규칙 로드
2. **엄격한 규칙 검증 (`validate_rule`)**: 영문 및 표준 CSS 기호(. # > 등)만 포함되었는지 확인 (자연어 슬래시나 한글 설명 엄격 차단)
3. **폴백 보정 (Fallback)**: 비어있거나 무효한 선택자는 안정적인 기본값(`div`, `span.spkrbld`, `p`)으로 대체
4. **통합 JSON 파일 빌드**: 4단계 텍스트 추출기가 일괄 참조할 수 있는 통합 딕셔너리로 저장

In [ ]:
import glob
import json
import os
import re
from pathlib import Path

# ---------------------------------------------------------------------------
# 1. 환경 설정 및 경로 지정
# ---------------------------------------------------------------------------
TARGET_YEAR = "2024"
RULES_INPUT_DIR = "./data/llm_rules_outputs/광역 회의록"
ALL_RULE_FILE = f"./data/to_text/llm_rules_outputs/consolidated_parsing_rules_{TARGET_YEAR}.json"

KOREAN_PATTERN = re.compile(r'[가-힣]')

FALLBACK_SELECTORS = {
    "agenda": "div",
    "speaker": "span.spkrbld",
    "utterance": "p"
}

print(f"규칙 폴더: {os.path.abspath(RULES_INPUT_DIR)}")
print(f"통합 출력: {os.path.abspath(ALL_RULE_FILE)}")

## 2. 규칙 검증 및 의회명 정규화 함수

In [ ]:
def validate_rule(rule_dict: dict) -> bool:
    """추출된 규칙이 유효한 순수 CSS 선택자인지 검사합니다."""
    for key in ("agenda", "speaker", "utterance"):
        val = rule_dict.get(key, "")
        if not isinstance(val, str) or not val.strip():
            return False
        # 한글 또는 자연어 설명('/') 포함 시 비정상 규칙으로 판정
        if KOREAN_PATTERN.search(val) or "/" in val:
            return False
    return True

def normalize_council_name(filename: str) -> str:
    """파일명에서 rule_, sample_ 등의 접두어를 제거하여 순수 의회명을 추출합니다."""
    stem = Path(filename).stem
    for prefix in ("rule_sample_", "rule_", "sample_"):
        if stem.startswith(prefix):
            stem = stem[len(prefix):]
    return stem

## 3. 규칙 통합 및 유효성 검증 실행

In [ ]:
rule_files = sorted(glob.glob(os.path.join(RULES_INPUT_DIR, "rule_*.json")))
if not rule_files:
    rule_files = sorted(glob.glob(os.path.join(RULES_INPUT_DIR, "**", "rule_*.json"), recursive=True))

print(f"검증 대상 규칙 파일 수: {len(rule_files)}개")

consolidated_rules = {}

# 기존 통합 파일이 존재할 경우 기존 규칙 보존
if os.path.exists(ALL_RULE_FILE):
    try:
        with open(ALL_RULE_FILE, "r", encoding="utf-8") as f:
            prev = json.load(f)
            if isinstance(prev, dict):
                consolidated_rules.update(prev)
                print(f"기존 통합 파일에서 {len(consolidated_rules)}개 규칙을 불러왔습니다.")
    except Exception as e:
        print(f"기존 규칙 로드 경고: {e}")

for file_path in rule_files:
    council_name = normalize_council_name(os.path.basename(file_path))
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        candidate = data
        if isinstance(data, dict) and council_name not in data and len(data) == 1:
            candidate = next(iter(data.values()))
            
        direct_rule = {
            key: candidate.get(key, "") for key in ("agenda", "speaker", "utterance")
        }
        
        # 텍스트 노드 특수 규칙 또는 일반 유효 CSS 선택자 확인
        is_text_node = direct_rule.get("speaker") == "__TEXT_NODE_SPEAKER__"
        if is_text_node or validate_rule(direct_rule):
            consolidated_rules[council_name] = direct_rule
            print(f"✓ [{council_name}] 규칙 등록: {direct_rule}")
        else:
            print(f"⚠️ [{council_name}] 규칙 유효성 부족 -> 기본 Fallback 적용")
            consolidated_rules[council_name] = dict(FALLBACK_SELECTORS)
    except Exception as exc:
        print(f"❌ [{council_name}] 파일 파싱 실패: {exc} -> 기본 Fallback 적용")
        consolidated_rules[council_name] = dict(FALLBACK_SELECTORS)

# 통합 규칙 파일 저장
os.makedirs(os.path.dirname(ALL_RULE_FILE) or ".", exist_ok=True)
with open(ALL_RULE_FILE, "w", encoding="utf-8") as f:
    json.dump(consolidated_rules, f, ensure_ascii=False, indent=2)

print(f"\n통합 완료! 총 {len(consolidated_rules)}개 의회 규칙이 저장되었습니다 -> {ALL_RULE_FILE}")

## 4. 최종 통합 규칙 확인 및 테이블 미리보기

In [ ]:
import pandas as pd

if os.path.exists(ALL_RULE_FILE):
    with open(ALL_RULE_FILE, "r", encoding="utf-8") as f:
        final_rules = json.load(f)
    df_rules = pd.DataFrame.from_dict(final_rules, orient="index")
    df_rules.index.name = "의회명"
    print("통합 규칙 요약표:")
    display(df_rules)
else:
    print("통합 규칙 파일이 존재하지 않습니다.")